# Credit Score Classification


In [ ]:
# Stage 2 Coding Part

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Visualization settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

COLOR_HIST = '#3498db'
COLOR_BOX = '#2ecc71'
COLOR_PIE = '#e74c3c'
COLOR_SCATTER = '#9b59b6'

print("All libraries loaded successfully!")

# 1. Load Data
df = pd.read_csv('train.csv')
print(f"Dataset shape: {df.shape}")

In [ ]:
# 2. Advanced Data Cleaning
df_cleaned = df.copy()

cols_drop = [c for c in ['ID', 'SSN', 'Name'] if c in df_cleaned.columns]
df_cleaned = df_cleaned.drop(columns=cols_drop)
df_cleaned = df_cleaned.drop_duplicates()

cols_to_fix = ['Age', 'Annual_Income', 'Outstanding_Debt', 'Monthly_Inhand_Salary']

print("Applying Advanced Data Recovery (Regex)...")
for col in cols_to_fix:
    if col in df_cleaned.columns:
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace('_', '', regex=False)
        df_cleaned[col] = df_cleaned[col].astype(str).str.replace(r'[^\d.-]', '', regex=True)
        df_cleaned[col] = pd.to_numeric(df_cleaned[col], errors='coerce')

df_cleaned.loc[df_cleaned['Age'] > 100, 'Age'] = np.nan
df_cleaned.loc[df_cleaned['Monthly_Inhand_Salary'] < 0, 'Monthly_Inhand_Salary'] = np.nan


In [ ]:
# 3. Missing Value Handling (Imputation)
numeric_cols = df_cleaned.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    df_cleaned[col].fillna(df_cleaned[col].median(), inplace=True)

categorical_cols = df_cleaned.select_dtypes(include=['object']).columns
for col in categorical_cols:
    mode_val = df_cleaned[col].mode()[0] if not df_cleaned[col].mode().empty else 'Unknown'
    df_cleaned[col].fillna(mode_val, inplace=True)

print(f"✓ Data Cleaned & Recovered. New Shape: {df_cleaned.shape}")

In [ ]:
# 4. Outlier Handling (IQR Clipping)
numeric_features = df_cleaned.select_dtypes(include=[np.number]).columns
for col in numeric_features:
    Q1 = df_cleaned[col].quantile(0.25)
    Q3 = df_cleaned[col].quantile(0.75)
    IQR = Q3 - Q1

    df_cleaned[col] = df_cleaned[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

print("✓ Outliers Handled (IQR Method)")


In [ ]:
# 5. Encoding
df_cleaned = df_cleaned[df_cleaned['Credit_Mix'] != '_']
df_cleaned = df_cleaned[df_cleaned['Payment_Behaviour'] != '!@9#%8']

if 'Payment_of_Min_Amount' in df_cleaned.columns:
    df_cleaned['Payment_of_Min_Amount'] = df_cleaned['Payment_of_Min_Amount'].map({'No': 0, 'Yes': 1})
if 'Credit_Mix' in df_cleaned.columns:
    df_cleaned['Credit_Mix'] = df_cleaned['Credit_Mix'].map({'Bad': -1, 'Standard': 0, 'Good': 1})

df_analysis = df_cleaned.copy()

df_cleaned = pd.get_dummies(df_cleaned, columns=['Occupation', 'Payment_Behaviour', 'Month'], drop_first=True)


In [ ]:
# 6. Scaling
scaler = StandardScaler()
num_cols = df_cleaned.select_dtypes(include=[np.number]).columns
df_cleaned[num_cols] = scaler.fit_transform(df_cleaned[num_cols])

print("✓ Preprocessing Complete!")


In [ ]:
# VIZ 1: Histogram
features_to_plot = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary',
                    'Credit_Utilization_Ratio', 'Outstanding_Debt']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Distribution of Numeric Features (Histogram)', fontsize=14, fontweight='bold')
axes = axes.flatten()
for i, col in enumerate(features_to_plot):
    if col in df_analysis.columns:
        sns.histplot(df_analysis[col], bins=25, kde=True, ax=axes[i], color=COLOR_HIST, edgecolor='white', linewidth=0.5)
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_facecolor('#f8f9fa')
for i in range(len(features_to_plot), len(axes)): axes[i].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# VIZ 2: Boxplot
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Boxplot - Outliers Analysis (After Handling)', fontsize=14, fontweight='bold')
axes = axes.flatten()
for i, col in enumerate(features_to_plot):
    if col in df_analysis.columns:
        sns.boxplot(y=df_analysis[col], ax=axes[i], color=COLOR_BOX, width=0.5)
        axes[i].set_title(col, fontweight='bold')
        axes[i].set_facecolor('#f8f9fa')
for i in range(len(features_to_plot), len(axes)): axes[i].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# VIZ 3: Heatmap
plt.figure(figsize=(12, 10))
numeric_df = df_analysis.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm', linewidths=0.5, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Heatmap', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()


In [ ]:
# VIZ 4: Categorical Plots
fig, ax = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor('white')

# Pie Chart
if 'Credit_Mix' in df_analysis.columns:
    colors_pie = ['#27ae60', '#f39c12', '#e74c3c']
    df_analysis['Credit_Mix'].value_counts().plot.pie(
        autopct='%1.1f%%', ax=ax[0], colors=colors_pie,
        textprops={'fontsize': 10, 'weight': 'bold'}
    )
    ax[0].set_title('Credit Mix Distribution', fontweight='bold', fontsize=12)
    ax[0].set_ylabel('')

# Count Plot
if 'Payment_of_Min_Amount' in df_analysis.columns:
    sns.countplot(x='Payment_of_Min_Amount', data=df_analysis, ax=ax[1], palette=['#3498db', '#2ecc71'])
    ax[1].set_title('Minimum Payment Status (0=No, 1=Yes)', fontweight='bold', fontsize=12)
    ax[1].set_xlabel('Payment Status', fontweight='bold')
    ax[1].set_ylabel('Count', fontweight='bold')
    ax[1].set_facecolor('#f8f9fa')

plt.tight_layout()
plt.show()

In [ ]:
# VIZ 5: Pair Plot
print("\nGenerating Pair Plot...")

pair_cols = ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Outstanding_Debt', 'Credit_Utilization_Ratio']

sns.pairplot(df_analysis[pair_cols],
             diag_kind='kde',
             kind='scatter',
             corner=True,
             plot_kws={'alpha': 0.6, 's': 50, 'edgecolor': 'white'},
             diag_kws={'shade': True, 'linewidth': 2, 'color': COLOR_SCATTER})
plt.suptitle('Pair Plot - Key Relationships', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

print("✓ All Visualizations Generated!")

In [ ]:
# Save cleaned dataset to CSV file
output_filename = 'cleaned_data.csv'
df_analysis.to_csv(output_filename, index=False)

print(f"✓ Cleaned dataset saved successfully!")
print(f"  - Filename: {output_filename}")
print(f"  - Rows: {df_analysis.shape[0]:,}")
print(f"  - Columns: {df_analysis.shape[1]}")
print(f"  - File size: {df_analysis.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Stage 3 Coding Part

In [ ]:

import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv('cleaned_data.csv')
df = df.drop_duplicates().reset_index(drop=True)

cols_to_drop = [col for col in ['Customer_ID', 'ID', 'SSN', 'Name'] if col in df.columns]
df = df.drop(columns=cols_to_drop)

numeric_like_cols = [
    'Age',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Num_Bank_Accounts',
    'Num_Credit_Card',
    'Interest_Rate',
    'Num_of_Loan',
    'Delay_from_due_date',
    'Num_of_Delayed_Payment',
    'Changed_Credit_Limit',
    'Num_Credit_Inquiries',
    'Outstanding_Debt',
    'Credit_Utilization_Ratio',
    'Total_EMI_per_month',
    'Amount_invested_monthly',
    'Monthly_Balance',
]

for col in numeric_like_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col].astype(str).str.replace(r'[^0-9.-]', '', regex=True), errors='coerce')

def credit_history_to_months(value: str) -> float:
    if pd.isna(value):
        return np.nan
    numbers = [int(x) for x in re.findall(r'\d+', str(value))]
    if len(numbers) >= 2:
        years, months = numbers[0], numbers[1]
    elif len(numbers) == 1:
        years, months = numbers[0], 0
    else:
        return np.nan
    return years * 12 + months

if 'Credit_History_Age' in df.columns:
    df['Credit_History_Age_Months'] = df['Credit_History_Age'].apply(credit_history_to_months)
    df.drop(columns=['Credit_History_Age'], inplace=True)

if 'Type_of_Loan' in df.columns:
    df['Loan_Types_Count'] = df['Type_of_Loan'].astype(str).apply(
        lambda x: len([loan.strip() for loan in x.split(',') if loan.strip()])
    )
    df['Has_Not_Specified_Loan'] = df['Type_of_Loan'].astype(str).str.contains('Not Specified', case=False).astype(int)
    df.drop(columns=['Type_of_Loan'], inplace=True)

if 'Payment_of_Min_Amount' in df.columns:
    df['Payment_of_Min_Amount'] = (
        df['Payment_of_Min_Amount']
        .replace({'Yes': 1, 'No': 0, 'NM': np.nan})
        .astype(float)
    )

if 'Credit_Mix' in df.columns:
    df['Credit_Mix'] = df['Credit_Mix'].replace({'Standard': 0, 'Good': 1, 'Bad': -1}).astype(float)

if 'Age' in df.columns:
    df.loc[(df['Age'] <= 0) | (df['Age'] > 100), 'Age'] = np.nan
if 'Monthly_Inhand_Salary' in df.columns:
    df.loc[df['Monthly_Inhand_Salary'] <= 0, 'Monthly_Inhand_Salary'] = np.nan


target_col = 'Credit_Score'
if target_col not in df.columns:
    raise ValueError("Credit_Score sutunu bulunamadi. Dogru veri setini kullandiginizdan emin olun.")

y = df[target_col].astype(str)
X = df.drop(columns=[target_col])


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
 )

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")
print(f"Class distribution (train):\n{y_train.value_counts(normalize=True)}")


numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
    ]
)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision_weighted',
    'recall': 'recall_weighted',
    'f1': 'f1_weighted',
    'roc_auc': 'roc_auc_ovr_weighted',
}

results = []
trained_pipelines = {}

print("\n--- Model training started ---\n")

for model_name, estimator in models.items():
    pipeline = Pipeline(steps=[('preprocess', preprocess), ('model', estimator)])

    cv_scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1,
    )

    pipeline.fit(X_train, y_train)
    trained_pipelines[model_name] = pipeline

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    y_train_proba = pipeline.predict_proba(X_train)
    y_test_proba = pipeline.predict_proba(X_test)

    train_metrics = {
        'accuracy': accuracy_score(y_train, y_train_pred),
        'precision': precision_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'roc_auc': roc_auc_score(y_train, y_train_proba, multi_class='ovr', average='weighted'),
    }

    test_metrics = {
        'accuracy': accuracy_score(y_test, y_test_pred),
        'precision': precision_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'f1': f1_score(y_test, y_test_pred, average='weighted', zero_division=0),
        'roc_auc': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='weighted'),
    }

    results.append({
        'Model': model_name,
        'CV Accuracy Mean': f"{cv_scores['test_accuracy'].mean():.4f} ± {cv_scores['test_accuracy'].std():.4f}",
        'CV Precision Mean': f"{cv_scores['test_precision'].mean():.4f} ± {cv_scores['test_precision'].std():.4f}",
        'CV Recall Mean': f"{cv_scores['test_recall'].mean():.4f} ± {cv_scores['test_recall'].std():.4f}",
        'CV F1 Mean': f"{cv_scores['test_f1'].mean():.4f} ± {cv_scores['test_f1'].std():.4f}",
        'CV ROC-AUC Mean': f"{cv_scores['test_roc_auc'].mean():.4f} ± {cv_scores['test_roc_auc'].std():.4f}",
        'Train Accuracy': train_metrics['accuracy'],
        'Train Precision': train_metrics['precision'],
        'Train Recall': train_metrics['recall'],
        'Train F1': train_metrics['f1'],
        'Train ROC-AUC': train_metrics['roc_auc'],
        'Test Accuracy': test_metrics['accuracy'],
        'Test Precision': test_metrics['precision'],
        'Test Recall': test_metrics['recall'],
        'Test F1': test_metrics['f1'],
        'Test ROC-AUC': test_metrics['roc_auc'],
    })

results_df = pd.DataFrame(results)
print('\n--- Cross-Validation Summary ---')
print(results_df[['Model', 'CV Accuracy Mean', 'CV Precision Mean', 'CV Recall Mean', 'CV F1 Mean', 'CV ROC-AUC Mean']])

print('\n--- Training Metrics ---')
print(results_df[['Model', 'Train Accuracy', 'Train Precision', 'Train Recall', 'Train F1', 'Train ROC-AUC']])

print('\n--- Test Metrics ---')
print(results_df[['Model', 'Test Accuracy', 'Test Precision', 'Test Recall', 'Test F1', 'Test ROC-AUC']])

best_model_name = results_df.sort_values('Test F1', ascending=False).iloc[0]['Model']
print(f"\nBest model based on Test F1: {best_model_name}")

results_df.to_csv('stage3_model_results.csv', index=False)
print("\nResults saved to stage3_model_results.csv")

In [ ]:
# Stage 4 Coding Part

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# XGBoost
try:
    from xgboost import XGBClassifier
except ImportError:
    !pip install xgboost -q
    from xgboost import XGBClassifier


print("All libraries loaded!")

In [ ]:
# Prepare X and y
target_col = 'Credit_Score'
y_raw = df[target_col].astype(str)
X = df.drop(columns=[target_col])


from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target Classes: {label_encoder.classes_}")
print(f"Encoded as: {list(range(len(label_encoder.classes_)))}")
print(f"Class distribution:\n{pd.Series(y_train).value_counts(normalize=True)}")

In [ ]:
# Preprocessing pipeline
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [col for col in X.columns if col not in numeric_features]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocess = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

print(f"Numeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
# NON-ENSEMBLE MODELS TRAINING

# Non-Ensemble Models
non_ensemble_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=12, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

# Cross-Validation Setup
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision_weighted', 'recall_weighted', 'f1_weighted']

non_ensemble_results = []
trained_non_ensemble = {}

X_train_pre = preprocess.fit_transform(X_train)
X_test_pre = preprocess.transform(X_test)

feature_names = numeric_features.copy()
if categorical_features:
    cat_encoder = preprocess.named_transformers_['cat'].named_steps['encoder']
    cat_feature_names = cat_encoder.get_feature_names_out(categorical_features).tolist()
    feature_names.extend(cat_feature_names)


print("NON-ENSEMBLE MODELS TRAINING")

print(f"\nCross-Validation: Stratified K-Fold (k=3)")
print(f"Evaluation Metrics: Accuracy, Precision, Recall, F1-Score\n")

for name, model in non_ensemble_models.items():
    print(f"\n{'─'*50}")
    print(f"Training: {name}")
    print(f"{'─'*50}")

    # Cross-validation
    cv_scores = cross_validate(model, X_train_pre, y_train, cv=cv, scoring=scoring, n_jobs=1)

    # Fit and predict
    model.fit(X_train_pre, y_train)
    trained_non_ensemble[name] = model

    y_pred = model.predict(X_test_pre)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    non_ensemble_results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    print(f"  CV Accuracy:  {cv_scores['test_accuracy'].mean():.4f} ± {cv_scores['test_accuracy'].std():.4f}")
    print(f"  CV F1-Score:  {cv_scores['test_f1_weighted'].mean():.4f} ± {cv_scores['test_f1_weighted'].std():.4f}")
    print(f"  Test Accuracy: {acc:.4f}")
    print(f"  Test F1-Score: {f1:.4f}")

In [ ]:
# ENSEMBLE MODELS TRAINING (BEFORE HYPERPARAMETER TUNING)

import time

ensemble_models = {
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=5, random_state=42,
                             use_label_encoder=False, eval_metric='mlogloss',
                             n_jobs=-1, tree_method='hist'),
}

ensemble_results_before = []
trained_ensemble = {}

print("ENSEMBLE MODELS (BEFORE HYPERPARAMETER TUNING)")

for name, model in ensemble_models.items():
    print(f"\n{'─'*50}")
    print(f"Training: {name}")
    print(f"{'─'*50}")

    start_time = time.time()

    model.fit(X_train_pre, y_train)
    trained_ensemble[name] = model

    y_pred = model.predict(X_test_pre)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    elapsed = time.time() - start_time

    ensemble_results_before.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")
    print(f"  Training time: {elapsed:.1f}s")

In [ ]:
# HYPERPARAMETER TUNING (RandomizedSearchCV)

param_grids = {
    'Random Forest': {
        'n_estimators': [200, 300, 400],  # Start from baseline
        'max_depth': [15, 20, 25, None],  # None = unlimited
        'min_samples_split': [2, 3],
        'min_samples_leaf': [1, 2],
    },
    'Gradient Boosting': {
        'n_estimators': [100, 150, 200],
        'max_depth': [3, 4, 5],
        'learning_rate': [0.05, 0.1, 0.15],
        'subsample': [0.8, 1.0],
    },
    'XGBoost': {
        'n_estimators': [100, 150, 200],  # Start from baseline
        'max_depth': [5, 6, 7],  # Start from baseline
        'learning_rate': [0.05, 0.1, 0.15],
        'subsample': [0.8, 1.0],
    }
}

# Base models
base_models = {
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss',
                             n_jobs=-1, tree_method='hist'),
}

tuned_models = {}
tuning_results = []


print("HYPERPARAMETER TUNING (RandomizedSearchCV)")

for name, model in base_models.items():
    print(f"\n{'─'*50}")
    print(f"Tuning: {name}")
    print(f"{'─'*50}")
    print(f"Parameter grid: {list(param_grids[name].keys())}")

    start_time = time.time()

    search = RandomizedSearchCV(
        model,
        param_grids[name],
        n_iter=10,
        cv=3,
        scoring='f1_weighted',
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train_pre, y_train)
    tuned_models[name] = search.best_estimator_

    elapsed = time.time() - start_time


    y_pred_tuned = search.best_estimator_.predict(X_test_pre)
    acc_tuned = accuracy_score(y_test, y_pred_tuned)
    prec_tuned = precision_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    rec_tuned = recall_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    f1_tuned = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)

    tuning_results.append({
        'Model': name,
        'Hyperparameters Tuned': ', '.join(param_grids[name].keys()),
        'Best Values': str(search.best_params_),
        'Best CV F1': search.best_score_
    })

    print(f"  Best parameters: {search.best_params_}")
    print(f"  Best CV F1: {search.best_score_:.4f}")
    print(f"  Test Accuracy:  {acc_tuned:.4f}")
    print(f"  Test Precision: {prec_tuned:.4f}")
    print(f"  Test Recall:    {rec_tuned:.4f}")
    print(f"  Test F1-Score:  {f1_tuned:.4f}")
    print(f"  Tuning time: {elapsed:.1f}s")

In [ ]:
# ENSEMBLE MODELS EVALUATION (AFTER HYPERPARAMETER TUNING)

ensemble_results_after = []

print("ENSEMBLE MODELS (AFTER HYPERPARAMETER TUNING)")


for name, model in tuned_models.items():
    print(f"\n{'─'*50}")
    print(f"{name} (Tuned)")
    print(f"{'─'*50}")

    y_pred = model.predict(X_test_pre)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    ensemble_results_after.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1
    })

    print(f"  Accuracy:  {acc:.4f}")
    print(f"  Precision: {prec:.4f}")
    print(f"  Recall:    {rec:.4f}")
    print(f"  F1-Score:  {f1:.4f}")

In [ ]:
# FEATURE IMPORTANCE - ALL ENSEMBLE MODELS COMPARISON


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (name, model) in enumerate(tuned_models.items()):
    importances = model.feature_importances_
    indices = np.argsort(importances)[-15:]  # Top 15

    colors = ['#3498db', '#2ecc71', '#e74c3c'][idx]
    axes[idx].barh(range(len(indices)), importances[indices], color=colors)
    axes[idx].set_yticks(range(len(indices)))
    axes[idx].set_yticklabels([feature_names[i] if i < len(feature_names) else f'feature_{i}' for i in indices])
    axes[idx].set_xlabel('Importance')
    axes[idx].set_title(f'{name}', fontweight='bold')
    axes[idx].set_facecolor('#f8f9fa')

plt.suptitle('Feature Importance Comparison (Top 15 Features)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance comparison saved to 'feature_importance_comparison.png'")

In [ ]:
# OVERALL MODEL COMPARISON (ALL 6 MODELS)


all_results = non_ensemble_results + ensemble_results_after
all_results_df = pd.DataFrame(all_results)

print("OVERALL MODEL COMPARISON (ALL 6 MODELS)")

all_results_df['Type'] = ['Non-Ensemble']*3 + ['Ensemble']*3

print("\n" + all_results_df[['Model', 'Type', 'Accuracy', 'Precision', 'Recall', 'F1-Score']].to_string(index=False))

display(all_results_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}).set_caption("Overall Model Performance Comparison"))

In [ ]:
# MODEL COMPARISON VISUALIZATION

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Bar chart comparison
ax1 = axes[0]
models_names = all_results_df['Model'].tolist()
x = np.arange(len(models_names))
width = 0.2

ax1.bar(x - width*1.5, all_results_df['Accuracy'], width, label='Accuracy', color='#3498db')
ax1.bar(x - width/2, all_results_df['Precision'], width, label='Precision', color='#2ecc71')
ax1.bar(x + width/2, all_results_df['Recall'], width, label='Recall', color='#e74c3c')
ax1.bar(x + width*1.5, all_results_df['F1-Score'], width, label='F1-Score', color='#9b59b6')

ax1.set_ylabel('Score', fontweight='bold')
ax1.set_title('Model Performance Comparison', fontweight='bold', fontsize=12)
ax1.set_xticks(x)
ax1.set_xticklabels(models_names, rotation=45, ha='right')
ax1.legend(loc='lower right')
ax1.set_ylim(0, 1)
ax1.set_facecolor('#f8f9fa')
ax1.axhline(y=0.5, color='gray', linestyle='--', alpha=0.3)

# Plot 2: F1-Score comparison
ax2 = axes[1]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12', '#1abc9c']
sorted_df = all_results_df.sort_values('F1-Score', ascending=True)
ax2.barh(sorted_df['Model'], sorted_df['F1-Score'], color=colors[:len(sorted_df)])
ax2.set_xlabel('F1-Score', fontweight='bold')
ax2.set_title('F1-Score Ranking', fontweight='bold', fontsize=12)
ax2.set_facecolor('#f8f9fa')
for i, (idx, row) in enumerate(sorted_df.iterrows()):
    ax2.text(row['F1-Score'] + 0.01, i, f"{row['F1-Score']:.4f}", va='center')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Model comparison chart saved to 'model_comparison.png'")

In [ ]:
# BEST AND WORST PERFORMING MODELS


best_model = all_results_df.loc[all_results_df['F1-Score'].idxmax()]
worst_model = all_results_df.loc[all_results_df['F1-Score'].idxmin()]

print("BEST PERFORMING MODEL")

print(f"  Model:     {best_model['Model']}")
print(f"  Type:      {best_model['Type']}")
print(f"  Accuracy:  {best_model['Accuracy']:.4f}")
print(f"  Precision: {best_model['Precision']:.4f}")
print(f"  Recall:    {best_model['Recall']:.4f}")
print(f"  F1-Score:  {best_model['F1-Score']:.4f}")


print("LEAST PERFORMING MODEL")

print(f"  Model:     {worst_model['Model']}")
print(f"  Type:      {worst_model['Type']}")
print(f"  Accuracy:  {worst_model['Accuracy']:.4f}")
print(f"  Precision: {worst_model['Precision']:.4f}")
print(f"  Recall:    {worst_model['Recall']:.4f}")
print(f"  F1-Score:  {worst_model['F1-Score']:.4f}")

improvement = ((best_model['F1-Score'] - worst_model['F1-Score']) / worst_model['F1-Score']) * 100
print(f"\n📊 Performance Gap: {improvement:.2f}% improvement from worst to best model")